# exp_rahul_002 — stage C on Kaggle: judge our arm with GLM-4-9B-0414 and score it
Free alternative to the Colab L4 stage. Before running: **Accelerator: GPU T4 x2**, **Internet: On**, and **Add Input**:
1. a dataset with the repo zip (`Steerability-Trust-issues-main.zip`, from GitHub → Code → Download ZIP; private, so download it yourself) — or attach the repo as a Kaggle dataset you maintain;
2. a dataset with `answers.jsonl` (both arms) and `judge_cache.jsonl` (exp001's verdicts) from the Drive run folder;
3. optionally the judge as a Kaggle Model (import `zai-org/GLM-4-9B-0414` revision `645b848`) so there is no download.

The judge runs in fp16 on the two T4s through `judge_local.py`; everything else is the team's scorer. ~20–40 min.

In [ ]:
ARM = "nullspace_v2"
SPLIT = "eval"
JUDGE_MODEL = "zai-org/GLM-4-9B-0414"   # or the mounted Kaggle Model path, e.g. "/kaggle/input/models/.../glm-4-9b-0414"

In [ ]:
import glob, os, zipfile, shutil
zips = sorted(glob.glob("/kaggle/input/**/*.zip", recursive=True))
repo = next((d for d in glob.glob("/kaggle/input/**/eval/scorer.py", recursive=True)), None)
if repo is None:
    z = next(z for z in zips if "Steerability" in os.path.basename(z))
    zipfile.ZipFile(z).extractall("/kaggle/working/repo"); repo = glob.glob("/kaggle/working/repo/**/eval/scorer.py", recursive=True)[0]
REPO = os.path.dirname(os.path.dirname(repo))
answers = glob.glob("/kaggle/input/**/answers.jsonl", recursive=True)[0]
cache_src = next(iter(glob.glob("/kaggle/input/**/judge_cache.jsonl", recursive=True)), None)
os.makedirs("/kaggle/working/run", exist_ok=True)
shutil.copy(answers, "/kaggle/working/run/answers.jsonl")
if cache_src: shutil.copy(cache_src, "/kaggle/working/run/judge_cache.jsonl")
judge = next(iter(glob.glob("/kaggle/input/**/glm*4*9b*/config.json", recursive=True)), None)
JUDGE = os.path.dirname(judge) if judge else JUDGE_MODEL
print("repo:", REPO); print("answers:", answers); print("cache:", cache_src); print("judge:", JUDGE)

In [ ]:
%pip install -q -r {REPO}/eval/requirements.txt 2>&1 | tail -1
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
!cd {REPO} && python experiments/exp_rahul_002_nullspace_gated/judge_local.py --rows data/v7_tidy/{SPLIT}.jsonl \
    --answers /kaggle/working/run/answers.jsonl --arm {ARM} --cache /kaggle/working/run/judge_cache.jsonl \
    --judge-model "{JUDGE}" --dtype float16 2>&1 | tee /kaggle/working/run/summary_{ARM}.txt

Download `/kaggle/working/run/summary_<arm>.txt` and `judge_cache.jsonl` (now including our verdicts) from the Output tab and copy them to the Drive run folder / `results.md`.